# DS4SE26 Week 3 — ARC Clustering for Group 4 / Lucene Codecs

This notebook is preconfigured for your new PC2 scratch folder:

`/scratch/hpc-prf-dssecs/group4`

Expected input files:

- `/scratch/hpc-prf-dssecs/group4/input/lucene-codecs-focused.rsf`
- `/scratch/hpc-prf-dssecs/group4/input/wca_uem.rsf`
- `/scratch/hpc-prf-dssecs/group4/input/wca_uemnm.rsf`
- `/scratch/hpc-prf-dssecs/group4/input/limbo_il.rsf`

Outputs will be written to:

`/scratch/hpc-prf-dssecs/group4/output`


## Cell 1 — Setup paths and caches
Run this first. It forces the notebook to use the `group4` scratch folder, not your home directory.


In [ ]:
import os
import sys
import site
from pathlib import Path

# =========================
# FIXED GROUP 4 SCRATCH PATH
# =========================
WORK_DIR = Path("/scratch/hpc-prf-dssecs/group4")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
CACHE_DIR = WORK_DIR / "cache"
SRC_DIR = WORK_DIR / "src"
LOG_DIR = WORK_DIR / "logs"
EVAL_DIR = WORK_DIR / "arcade_eval"

for d in [WORK_DIR, INPUT_DIR, OUTPUT_DIR, CACHE_DIR, SRC_DIR, LOG_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Keep all model/cache files on scratch
os.environ["HF_HOME"] = str(CACHE_DIR / "hf_cache")
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "hf_cache" / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR / "hf_cache" / "transformers")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Fix PC2 user-local Python script path warnings
USER_BIN = Path.home() / ".local" / "bin"
os.environ["PATH"] = f"{USER_BIN}:{os.environ.get('PATH', '')}"

USER_SITE = site.getusersitepackages()
if USER_SITE not in sys.path:
    sys.path.insert(0, USER_SITE)

print("WORK_DIR:", WORK_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("HF_HOME:", os.environ["HF_HOME"])
print("Python:", sys.executable)
print("User site:", USER_SITE)
print("PATH starts with:", os.environ["PATH"].split(":")[:3])


## Cell 2 — Optional install required packages
Run this once. If imports still fail after install, restart the kernel and run again from Cell 1.


In [ ]:
import sys
import subprocess

packages = [
    "numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
    "torch", "transformers", "accelerate", "sentence-transformers", "huggingface_hub"
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "-q"] + packages)

print("Install/check complete. If the next cell fails to import, restart kernel once.")


## Cell 3 — Imports and warnings


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import re
import sys
import time
import json
import csv
import math
import shutil
import getpass
import subprocess
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

print("Imports OK")
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))


## Cell 4 — Rename input files if they still contain spaces
This handles old filenames like `wca uem.rsf` and renames them to notebook-safe names.


In [ ]:
rename_map = {
    "wca uem.rsf": "wca_uem.rsf",
    "wca uemnm.rsf": "wca_uemnm.rsf",
    "limbo il.rsf": "limbo_il.rsf",
    "Lucene_codecs_focused.rsf": "lucene-codecs-focused.rsf",
}

for old, new in rename_map.items():
    old_path = INPUT_DIR / old
    new_path = INPUT_DIR / new
    if old_path.exists() and not new_path.exists():
        old_path.rename(new_path)
        print(f"Renamed {old} -> {new}")

print("Input folder contents:")
for p in sorted(INPUT_DIR.glob("*")):
    print(" -", p.name, p.stat().st_size, "bytes")


## Cell 5 — Configuration
This cell contains all fixed paths and parameters.


In [ ]:
# Required input files
FOCUSED_DEPENDENCY_RSF = INPUT_DIR / "lucene-codecs-focused.rsf"
BASELINE_CLUSTER_FILES = {
    "WCA_UEM": INPUT_DIR / "wca_uem.rsf",
    "WCA_UEMNM": INPUT_DIR / "wca_uemnm.rsf",
    "LIMBO_IL": INPUT_DIR / "limbo_il.rsf",
}

# Week 3 assigned embedding model for Groups 4/9/14
EMBEDDING_MODEL_NAME = "Qodo/Qodo-Embed-1-7B"

# Match this with the Lucene version used for the dependency RSF.
LUCENE_GIT_TAG = "releases/lucene/9.10.0"

# ARC settings
ALPHA = 0.50
TARGET_NUM_CLUSTERS = 10
BATCH_SIZE = 2
MAX_TOKENS = 8192

RUN_NAME = f"ARC_Qodo_alpha{str(ALPHA).replace('.', '_')}_k{TARGET_NUM_CLUSTERS}"
ARC_RSF_OUT = OUTPUT_DIR / f"{RUN_NAME}_clusters.rsf"
ARC_CSV_OUT = OUTPUT_DIR / f"{RUN_NAME}_clusters.csv"
COMPARISON_CSV_OUT = OUTPUT_DIR / f"{RUN_NAME}_python_comparison_sanity.csv"
SUMMARY_OUT = OUTPUT_DIR / f"{RUN_NAME}_summary.txt"

print("Model:", EMBEDDING_MODEL_NAME)
print("Lucene tag:", LUCENE_GIT_TAG)
print("ALPHA:", ALPHA)
print("TARGET_NUM_CLUSTERS:", TARGET_NUM_CLUSTERS)
print("ARC output:", ARC_RSF_OUT)


## Cell 6 — Check input files
This must pass before you continue.


In [ ]:
missing = []
empty = []

all_required = [FOCUSED_DEPENDENCY_RSF] + list(BASELINE_CLUSTER_FILES.values())
for path in all_required:
    if not path.exists():
        missing.append(str(path))
    elif path.stat().st_size == 0:
        empty.append(str(path))

if missing:
    print("Missing files:")
    for m in missing:
        print(" -", m)
    raise FileNotFoundError("Missing required files. Fix input folder and rerun this cell.")

if empty:
    print("Empty files:")
    for e in empty:
        print(" -", e)
    raise RuntimeError("Some required files are 0 bytes. Replace them with correct files.")

print("All required files found and non-empty.")
for p in all_required:
    print(p, "|", p.stat().st_size, "bytes")

print("
First 5 lines of focused dependency RSF:")
with open(FOCUSED_DEPENDENCY_RSF, "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.rstrip())


## Cell 7 — Hugging Face token
Paste your token into the hidden prompt. It will not be printed.


In [ ]:
if not os.environ.get("HF_TOKEN"):
    token = getpass.getpass("Paste HF_TOKEN: ").strip()
    if token:
        os.environ["HF_TOKEN"] = token

HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF_TOKEN exists:", bool(HF_TOKEN))
if HF_TOKEN:
    print("Token prefix:", HF_TOKEN[:6])

try:
    from huggingface_hub import login
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face login OK")
except Exception as e:
    print("HF login warning:", repr(e))


## Cell 8 — Clone Apache Lucene source


In [ ]:
LUCENE_REPO_DIR = SRC_DIR / "lucene"

if not LUCENE_REPO_DIR.exists():
    print("Cloning Apache Lucene:", LUCENE_GIT_TAG)
    cmd = [
        "git", "clone", "--depth", "1", "--branch", LUCENE_GIT_TAG,
        "https://github.com/apache/lucene.git", str(LUCENE_REPO_DIR)
    ]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        print("Tag clone failed. Trying default branch clone.")
        if LUCENE_REPO_DIR.exists():
            shutil.rmtree(LUCENE_REPO_DIR)
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/apache/lucene.git", str(LUCENE_REPO_DIR)])
else:
    print("Lucene source already exists:", LUCENE_REPO_DIR)

SOURCE_ROOT_CANDIDATES = [
    LUCENE_REPO_DIR / "lucene" / "codecs" / "src" / "java",
    LUCENE_REPO_DIR / "lucene" / "codecs" / "src" / "main" / "java",
    LUCENE_REPO_DIR / "codecs" / "src" / "java",
    LUCENE_REPO_DIR / "codecs" / "src" / "main" / "java",
]

SOURCE_ROOT = None
for candidate in SOURCE_ROOT_CANDIDATES:
    if candidate.exists():
        SOURCE_ROOT = candidate
        break

if SOURCE_ROOT is None:
    raise FileNotFoundError("Lucene codecs source root not found. Checked: " + str(SOURCE_ROOT_CANDIDATES))

print("SOURCE_ROOT:", SOURCE_ROOT)


## Cell 9 — Read Java source files from Lucene Codecs


In [ ]:
def extract_package(java_text):
    match = re.search(r"^\s*package\s+([a-zA-Z0-9_.]+)\s*;", java_text, flags=re.MULTILINE)
    return match.group(1) if match else None

def java_file_to_fqn(java_file):
    text = java_file.read_text(encoding="utf-8", errors="ignore")
    package = extract_package(text)
    if package:
        return f"{package}.{java_file.stem}"
    rel = java_file.relative_to(SOURCE_ROOT).with_suffix("")
    return ".".join(rel.parts)

java_files = sorted(SOURCE_ROOT.rglob("*.java"))
source_records = []

for jf in java_files:
    text = jf.read_text(encoding="utf-8", errors="ignore")
    fqn = java_file_to_fqn(jf)
    if fqn.startswith("org.apache.lucene.codecs"):
        source_records.append({"entity": fqn, "path": str(jf), "text": text})

if not source_records:
    raise RuntimeError("No org.apache.lucene.codecs Java files found.")

entities = [r["entity"] for r in source_records]
texts = [r["text"] for r in source_records]
paths = [r["path"] for r in source_records]

print("Java files used:", len(source_records))
print("Example entity:", entities[0])


## Cell 10 — Read focused dependency RSF


In [ ]:
def normalize_entity_name(x):
    x = x.strip().strip('"').strip("'").strip(",")
    x = x.replace("/", ".").replace("\", ".")
    if x.endswith(".class"):
        x = x[:-6]
    x = x.replace("$", ".")
    return x

def parse_dependency_rsf(path):
    edges = []
    keyword_counts = Counter()
    sample_lines = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            raw = line.strip()
            if not raw or raw.startswith("#"):
                continue
            parts = raw.split()
            if not parts:
                continue
            keyword_counts[parts[0].lower()] += 1
            if len(sample_lines) < 5:
                sample_lines.append(raw)
            if len(parts) >= 3 and parts[0].lower() == "depends":
                edges.append((normalize_entity_name(parts[1]), normalize_entity_name(parts[2])))
    return edges, keyword_counts, sample_lines

dependency_edges, keyword_counts, sample_lines = parse_dependency_rsf(FOCUSED_DEPENDENCY_RSF)

print("Keyword counts:", keyword_counts)
print("Sample lines:")
for line in sample_lines:
    print(" ", line)

if not dependency_edges:
    raise RuntimeError("No 'depends Source Target' lines found. This must be the filtered dependency RSF from Week 1.")

rsf_entities = set()
for s, t in dependency_edges:
    rsf_entities.add(s)
    rsf_entities.add(t)

overlap_with_rsf = set(entities) & rsf_entities

print("Dependency edges:", len(dependency_edges))
print("Unique RSF entities:", len(rsf_entities))
print("Source entities overlapping with RSF:", len(overlap_with_rsf), "/", len(entities))
print("Example edge:", dependency_edges[0])

if len(overlap_with_rsf) < max(5, 0.20 * len(entities)):
    print("WARNING: Low overlap. Source Lucene version may not match Week 1 RSF version.")


## Cell 11 — Prepare texts for embedding


In [ ]:
def clean_code_for_embedding(entity, text):
    return f"Java source file: {entity}

{text}"

embedding_texts = [clean_code_for_embedding(e, t) for e, t in zip(entities, texts)]
print("Texts to embed:", len(embedding_texts))
print("First text characters:", len(embedding_texts[0]))


## Cell 12 — Load Qodo embedding model with PC2 compatibility patches


In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoConfig

try:
    del hf_model
except Exception:
    pass

torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

embedding_backend = "transformers"
st_model = None

tokenizer = AutoTokenizer.from_pretrained(
    EMBEDDING_MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
    cache_dir=str(CACHE_DIR / "hf_cache"),
)

config = AutoConfig.from_pretrained(
    EMBEDDING_MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
    cache_dir=str(CACHE_DIR / "hf_cache"),
)

# Patch for Qodo/Qwen remote code compatibility
if not hasattr(config, "rope_theta"):
    config.rope_theta = 10000.0
    print("Patched config.rope_theta =", config.rope_theta)
if not hasattr(config, "attention_dropout"):
    config.attention_dropout = 0.0
    print("Patched config.attention_dropout =", config.attention_dropout)

config._attn_implementation = "eager"
config.use_cache = False

model_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

try:
    hf_model = AutoModel.from_pretrained(
        EMBEDDING_MODEL_NAME,
        config=config,
        token=HF_TOKEN,
        trust_remote_code=True,
        dtype=model_dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
        cache_dir=str(CACHE_DIR / "hf_cache"),
    )
except TypeError:
    hf_model = AutoModel.from_pretrained(
        EMBEDDING_MODEL_NAME,
        config=config,
        token=HF_TOKEN,
        trust_remote_code=True,
        torch_dtype=model_dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
        cache_dir=str(CACHE_DIR / "hf_cache"),
    )

hf_model.eval()
hf_model.config.use_cache = False

print("Loaded successfully.")
print("First model parameter device:", next(hf_model.parameters()).device)


## Cell 13 — Compute embeddings


In [ ]:
def last_token_pool(last_hidden_states, attention_mask):
    left_padding = bool((attention_mask[:, -1].sum() == attention_mask.shape[0]).item())
    if left_padding:
        return last_hidden_states[:, -1]
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

def embed_with_transformers(text_list):
    all_embeddings = []
    first_device = next(hf_model.parameters()).device
    print("First model device:", first_device)

    with torch.no_grad():
        for start in range(0, len(text_list), BATCH_SIZE):
            batch = text_list[start:start + BATCH_SIZE]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=MAX_TOKENS, return_tensors="pt")
            inputs = {k: v.to(first_device) for k, v in inputs.items()}
            outputs = hf_model(**inputs, use_cache=False, return_dict=True)
            pooled = last_token_pool(outputs.last_hidden_state, inputs["attention_mask"])
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            all_embeddings.append(pooled.detach().float().cpu().numpy())
            print(f"Embedded {min(start + BATCH_SIZE, len(text_list))}/{len(text_list)} files")
    return np.vstack(all_embeddings).astype(np.float32)

start_time = time.time()
embeddings = embed_with_transformers(embedding_texts)
elapsed = time.time() - start_time

embeddings_path = OUTPUT_DIR / f"{RUN_NAME}_embeddings.npy"
np.save(embeddings_path, embeddings)

print("Embeddings shape:", embeddings.shape)
print(f"Embedding time: {elapsed/60:.2f} minutes")
print("Saved:", embeddings_path)


## Cell 14 — Semantic similarity matrix


In [ ]:
semantic_similarity = cosine_similarity(embeddings)
semantic_similarity = (semantic_similarity + 1.0) / 2.0
semantic_similarity = np.clip(semantic_similarity, 0.0, 1.0)
np.fill_diagonal(semantic_similarity, 1.0)

semantic_path = OUTPUT_DIR / f"{RUN_NAME}_semantic_similarity_matrix.npy"
np.save(semantic_path, semantic_similarity)

print("Semantic matrix shape:", semantic_similarity.shape)
print("Semantic min/max:", semantic_similarity.min(), semantic_similarity.max())
print("Saved:", semantic_path)


## Cell 15 — Structural similarity matrix


In [ ]:
entity_to_idx = {e: i for i, e in enumerate(entities)}
neighbors = {e: set() for e in entities}

for src, tgt in dependency_edges:
    if src in entity_to_idx:
        neighbors[src].add(tgt)
    if tgt in entity_to_idx:
        neighbors[tgt].add(src)

n = len(entities)
structural_similarity = np.zeros((n, n), dtype=np.float32)

for i, e1 in enumerate(entities):
    n1 = neighbors[e1]
    for j, e2 in enumerate(entities):
        if i == j:
            structural_similarity[i, j] = 1.0
            continue
        n2 = neighbors[e2]
        union = n1 | n2
        structural_similarity[i, j] = 0.0 if not union else len(n1 & n2) / len(union)

structural_similarity = np.clip(structural_similarity, 0.0, 1.0)
np.fill_diagonal(structural_similarity, 1.0)

structural_path = OUTPUT_DIR / f"{RUN_NAME}_structural_similarity_matrix.npy"
np.save(structural_path, structural_similarity)

print("Structural matrix shape:", structural_similarity.shape)
print("Structural min/max:", structural_similarity.min(), structural_similarity.max())
print("Non-zero off-diagonal structural entries:", np.count_nonzero(structural_similarity) - n)
print("Saved:", structural_path)


## Cell 16 — Combine matrices and create distance matrix


In [ ]:
combined_similarity = (ALPHA * structural_similarity) + ((1.0 - ALPHA) * semantic_similarity)
combined_similarity = np.clip(combined_similarity, 0.0, 1.0)
np.fill_diagonal(combined_similarity, 1.0)

distance_matrix = 1.0 - combined_similarity
distance_matrix = np.clip(distance_matrix, 0.0, 1.0)
np.fill_diagonal(distance_matrix, 0.0)

combined_path = OUTPUT_DIR / f"{RUN_NAME}_combined_similarity_matrix.npy"
distance_path = OUTPUT_DIR / f"{RUN_NAME}_distance_matrix.npy"
np.save(combined_path, combined_similarity)
np.save(distance_path, distance_matrix)

print("Combined min/max:", combined_similarity.min(), combined_similarity.max())
print("Distance min/max:", distance_matrix.min(), distance_matrix.max())
print("Saved combined:", combined_path)
print("Saved distance:", distance_path)


## Cell 17 — Agglomerative clustering and ARC RSF output


In [ ]:
try:
    clusterer = AgglomerativeClustering(n_clusters=TARGET_NUM_CLUSTERS, metric="precomputed", linkage="average")
except TypeError:
    clusterer = AgglomerativeClustering(n_clusters=TARGET_NUM_CLUSTERS, affinity="precomputed", linkage="average")

labels = clusterer.fit_predict(distance_matrix)

cluster_df = pd.DataFrame({"entity": entities, "cluster_id": labels, "source_path": paths}).sort_values(["cluster_id", "entity"])
cluster_df.to_csv(ARC_CSV_OUT, index=False)

with open(ARC_RSF_OUT, "w", encoding="utf-8") as f:
    for _, row in cluster_df.iterrows():
        f.write(f"contain ARC_Cluster_{int(row['cluster_id'])} {row['entity']}
")

print("ARC RSF saved:", ARC_RSF_OUT)
print("ARC CSV saved:", ARC_CSV_OUT)
print("Cluster sizes:")
print(cluster_df.groupby("cluster_id").size().reset_index(name="size").to_string(index=False))


## Cell 18 — Cluster size plot


In [ ]:
cluster_sizes = cluster_df.groupby("cluster_id").size().reset_index(name="size")

plt.figure(figsize=(10, 5))
plt.bar(cluster_sizes["cluster_id"].astype(str), cluster_sizes["size"])
plt.xlabel("ARC Cluster ID")
plt.ylabel("Number of Java files")
plt.title(f"ARC Cluster Sizes | alpha={ALPHA}, k={TARGET_NUM_CLUSTERS}")
plt.tight_layout()

plot_path = OUTPUT_DIR / f"{RUN_NAME}_cluster_sizes.png"
plt.savefig(plot_path, dpi=200)
plt.show()

print("Saved plot:", plot_path)


## Cell 19 — Python sanity comparison with Week 2 outputs
This is not the official ARCADE `a2a/cvg`, but it checks entity overlap and similarity.


In [ ]:
def parse_cluster_rsf(path):
    mapping = {}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 3 and parts[0].lower() in {"contain", "contains"}:
                cluster = parts[1]
                entity = normalize_entity_name(parts[2])
                mapping[entity] = cluster
    return mapping

cluster_maps = {"ARC_QODO": {row["entity"]: f"ARC_Cluster_{int(row['cluster_id'])}" for _, row in cluster_df.iterrows()}}

for label, path in BASELINE_CLUSTER_FILES.items():
    cluster_maps[label] = parse_cluster_rsf(path)
    print(f"{label}: {len(cluster_maps[label])} entities parsed")

def compare_cluster_maps(name_a, map_a, name_b, map_b):
    common = sorted(set(map_a.keys()) & set(map_b.keys()))
    if len(common) < 2:
        return {
            "architecture_a": name_a,
            "architecture_b": name_b,
            "entities_a": len(map_a),
            "entities_b": len(map_b),
            "common_entities": len(common),
            "clusters_a_common": None,
            "clusters_b_common": None,
            "adjusted_rand_index": None,
            "normalized_mutual_info": None,
            "note": "Too few common entities for meaningful comparison",
        }

    labels_a_raw = [map_a[e] for e in common]
    labels_b_raw = [map_b[e] for e in common]

    def encode(labels_raw):
        lookup = {}
        encoded = []
        for x in labels_raw:
            if x not in lookup:
                lookup[x] = len(lookup)
            encoded.append(lookup[x])
        return encoded, lookup

    labels_a, lookup_a = encode(labels_a_raw)
    labels_b, lookup_b = encode(labels_b_raw)

    return {
        "architecture_a": name_a,
        "architecture_b": name_b,
        "entities_a": len(map_a),
        "entities_b": len(map_b),
        "common_entities": len(common),
        "clusters_a_common": len(lookup_a),
        "clusters_b_common": len(lookup_b),
        "adjusted_rand_index": adjusted_rand_score(labels_a, labels_b),
        "normalized_mutual_info": normalized_mutual_info_score(labels_a, labels_b),
        "note": "Python sanity check only; use ARCADE a2a/cvg for official evaluation",
    }

names = list(cluster_maps.keys())
comparison_rows = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        comparison_rows.append(compare_cluster_maps(names[i], cluster_maps[names[i]], names[j], cluster_maps[names[j]]))

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(COMPARISON_CSV_OUT, index=False)

print("Comparison saved:", COMPARISON_CSV_OUT)
comparison_df


## Cell 20 — Write summary file


In [ ]:
summary_text = f"""
DS4SE26 Week 3 ARC Run Summary
==============================

Working directory:
{WORK_DIR}

Input files:
Focused dependency RSF: {FOCUSED_DEPENDENCY_RSF}
WCA_UEM: {BASELINE_CLUSTER_FILES['WCA_UEM']}
WCA_UEMNM: {BASELINE_CLUSTER_FILES['WCA_UEMNM']}
LIMBO_IL: {BASELINE_CLUSTER_FILES['LIMBO_IL']}

Model:
{EMBEDDING_MODEL_NAME}

Lucene source tag/branch:
{LUCENE_GIT_TAG}

ARC settings:
ALPHA = {ALPHA}
TARGET_NUM_CLUSTERS = {TARGET_NUM_CLUSTERS}
BATCH_SIZE = {BATCH_SIZE}
MAX_TOKENS = {MAX_TOKENS}

Counts:
Java source files used = {len(entities)}
Focused RSF dependency edges = {len(dependency_edges)}
Unique RSF entities = {len(rsf_entities)}
Source entities overlapping with focused RSF = {len(overlap_with_rsf)}

Outputs:
ARC RSF = {ARC_RSF_OUT}
ARC CSV = {ARC_CSV_OUT}
Embeddings = {embeddings_path}
Semantic matrix = {semantic_path}
Structural matrix = {structural_path}
Combined matrix = {combined_path}
Distance matrix = {distance_path}
Comparison sanity CSV = {COMPARISON_CSV_OUT}
Cluster size plot = {plot_path}

Important:
The Python comparison table is only a sanity check.
For official Week 3 evaluation, also run ARCADE a2a and cvg using:
- ARC output
- wca_uem.rsf
- wca_uemnm.rsf
- limbo_il.rsf
"""

SUMMARY_OUT.write_text(summary_text, encoding="utf-8")
print("Summary saved:", SUMMARY_OUT)
print(summary_text)


## Cell 21 — Prepare ARCADE a2a/cvg evaluation folder
This creates `/scratch/hpc-prf-dssecs/group4/arcade_eval` and copies the final RSF files there.


In [ ]:
TOOLS_DIR = EVAL_DIR / "tools"
RSF_DIR = EVAL_DIR / "rsf"
RESULTS_DIR = EVAL_DIR / "results"

for d in [TOOLS_DIR, RSF_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Copy files for evaluation
shutil.copy2(ARC_RSF_OUT, RSF_DIR / "arc_qodo.rsf")
shutil.copy2(BASELINE_CLUSTER_FILES["WCA_UEM"], RSF_DIR / "wca_uem.rsf")
shutil.copy2(BASELINE_CLUSTER_FILES["WCA_UEMNM"], RSF_DIR / "wca_uemnm.rsf")
shutil.copy2(BASELINE_CLUSTER_FILES["LIMBO_IL"], RSF_DIR / "limbo_il.rsf")

print("ARCADE eval folder:", EVAL_DIR)
print("Upload a2a/cvg jar files here:", TOOLS_DIR)
print("RSF files:")
for p in sorted(RSF_DIR.glob("*.rsf")):
    print(" -", p.name, p.stat().st_size, "bytes")


## Cell 22 — Detect a2a/cvg tools and generate command script
Upload your ARCADE a2a and cvg `.jar` files into the `tools` folder before running this cell.


In [ ]:
from itertools import combinations

all_jars = list(TOOLS_DIR.glob("*.jar"))
print("JAR files found:")
for jar in all_jars:
    print(" -", jar.name)

a2a_candidates = [j for j in all_jars if "a2a" in j.name.lower()]
cvg_candidates = [j for j in all_jars if "cvg" in j.name.lower() or "coverage" in j.name.lower()]

if not a2a_candidates:
    raise FileNotFoundError(f"No a2a jar found in {TOOLS_DIR}. Rename/upload so filename contains 'a2a'.")
if not cvg_candidates:
    raise FileNotFoundError(f"No cvg jar found in {TOOLS_DIR}. Rename/upload so filename contains 'cvg'.")

A2A_JAR = a2a_candidates[0]
CVG_JAR = cvg_candidates[0]
print("A2A_JAR:", A2A_JAR)
print("CVG_JAR:", CVG_JAR)

rsf_files = {
    "ARC_QODO": RSF_DIR / "arc_qodo.rsf",
    "WCA_UEM": RSF_DIR / "wca_uem.rsf",
    "WCA_UEMNM": RSF_DIR / "wca_uemnm.rsf",
    "LIMBO_IL": RSF_DIR / "limbo_il.rsf",
}
comparison_pairs = list(combinations(rsf_files.items(), 2))

commands_file = RESULTS_DIR / "arcade_eval_commands.sh"
with open(commands_file, "w", encoding="utf-8") as f:
    f.write("#!/bin/bash
set -e

")
    f.write(f"cd {EVAL_DIR}

")
    for (name_a, path_a), (name_b, path_b) in comparison_pairs:
        pair_name = f"{name_a}_VS_{name_b}"
        a2a_out = RESULTS_DIR / f"{pair_name}_a2a.txt"
        cvg_out = RESULTS_DIR / f"{pair_name}_cvg.txt"
        f.write(f"echo 'A2A: {pair_name}'
")
        f.write(f"java -jar {A2A_JAR} {path_a} {path_b} > {a2a_out} 2>&1

")
        f.write(f"echo 'CVG: {pair_name}'
")
        f.write(f"java -jar {CVG_JAR} {path_a} {path_b} > {cvg_out} 2>&1

")

commands_file.chmod(0o755)
print("Command script created:", commands_file)
print(commands_file.read_text())


## Cell 23 — Run ARCADE a2a/cvg comparisons
Run this only after Cell 22 succeeds.


In [ ]:
result = subprocess.run(["bash", str(commands_file)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)
print("Return code:", result.returncode)

print("Generated result files:")
for p in sorted(RESULTS_DIR.glob("*.txt")):
    print(" -", p.name, p.stat().st_size, "bytes")

if result.returncode != 0:
    print("One of the commands failed. Open the result text files to see the exact ARCADE usage error.")


## Cell 24 — Combine a2a/cvg result files into one summary


In [ ]:
combined_eval_summary = RESULTS_DIR / "week3_arcade_a2a_cvg_summary.txt"
with open(combined_eval_summary, "w", encoding="utf-8") as out:
    out.write("Week 3 ARCADE a2a/cvg Evaluation Summary
")
    out.write("=" * 60 + "

")
    for p in sorted(RESULTS_DIR.glob("*.txt")):
        if p.name == combined_eval_summary.name:
            continue
        out.write("
" + "=" * 80 + "
")
        out.write(p.name + "
")
        out.write("=" * 80 + "
")
        out.write(p.read_text(encoding="utf-8", errors="ignore"))
        out.write("

")

print("Combined summary saved:", combined_eval_summary)
print(combined_eval_summary.read_text(encoding="utf-8", errors="ignore")[:3000])
